In [50]:
# -------- LLM (GGUF via CTransformers) --------
from langchain_community.llms import CTransformers

# -------- Embeddings --------
from langchain_community.embeddings import HuggingFaceEmbeddings

# -------- Vector Store (Pinecone) --------
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

# -------- Document Loaders --------
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

# -------- Text Splitter --------
from langchain_text_splitters import RecursiveCharacterTextSplitter

# -------- Prompt Templates --------
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate

# -------- Retrieval Chain (NEW Modern Way) --------
from langchain_classic.chains import create_retrieval_chain
from dotenv import load_dotenv
import os


In [29]:
def load_pdf():
    loader = PyPDFLoader("data/The_GALE_ENCYCLOPEDIA_of_MEDICINE_SECOND.pdf")
    documents = loader.load()
    return documents

extracted_data = load_pdf()

In [32]:
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(
              chunk_size = 500 , chunk_overlap = 20
    )
    texts = text_splitter.split_documents(extracted_data)
    return texts

text_chunks = text_split(extracted_data)
len(text_chunks)

6972

In [33]:
def load_embedding_model():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

embeddings = load_embedding_model()

/var/folders/4l/zdggcq557j38h0s7ttxch25h0000gn/T/ipykernel_47105/2390791927.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2183.59it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [52]:
load_dotenv()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
pc = Pinecone(api_key= PINECONE_API_KEY)
vectorstore = PineconeVectorStore.from_documents(documents=text_chunks, embedding=embeddings, index_name="medical-chatbot")